# Análise de Sentimentos com TF-IDF e Regressão Logística
## Google Colab | Trabalho Acadêmico

In [ ]:
!pip install -q sklearn matplotlib pandas tqdm

import os
import tarfile
import urllib.request
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from tqdm import tqdm

## Questão 1 — Download, carregamento e TF-IDF

In [ ]:
url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
arquivo = "aclImdb_v1.tar.gz"

if not os.path.exists(arquivo):
    urllib.request.urlretrieve(url, arquivo)

if not os.path.exists("aclImdb"):
    with tarfile.open(arquivo, "r:gz") as tar:
        tar.extractall()

def carregar_dados(diretorio):
    textos = []
    labels = []
    for categoria in ['pos', 'neg']:
        caminho = os.path.join(diretorio, categoria)
        for arquivo in tqdm(os.listdir(caminho), desc=f"Carregando {categoria}"):
            with open(os.path.join(caminho, arquivo), encoding="utf-8") as f:
                textos.append(f.read())
                labels.append(1 if categoria == 'pos' else 0)
    return pd.DataFrame({'texto': textos, 'sentimento': labels})

dados = carregar_dados('aclImdb/train')

vetorizador = TfidfVectorizer(stop_words='english', max_features=10000)
x_tfidf = vetorizador.fit_transform(dados['texto'])
y = dados['sentimento']

## Questão 2 — Maiores e menores valores de TF-IDF

In [ ]:
media_tfidf = np.asarray(x_tfidf.mean(axis=0)).flatten()
features = vetorizador.get_feature_names_out()

df_tfidf = pd.DataFrame({'feature': features, 'media_tfidf': media_tfidf})

maiores = df_tfidf.sort_values(by='media_tfidf', ascending=False).head(10)
print("10 maiores TF-IDF:
", maiores)

menores = df_tfidf.sort_values(by='media_tfidf').head(10)
print("
10 menores TF-IDF:
", menores)

## Questão 3 — Modelo de Regressão Logística e Precisão

In [ ]:
modelo = LogisticRegression(max_iter=1000)
modelo.fit(x_tfidf, y)

y_pred = modelo.predict(x_tfidf)
precisao = accuracy_score(y, y_pred)
print(f"Precisão do modelo: {precisao:.4f}")

## Questão 4 — Análise dos coeficientes do modelo

In [ ]:
coeficientes = modelo.coef_.flatten()
df_coef = pd.DataFrame({'feature': features, 'coeficiente': coeficientes})

maiores_coef = df_coef.sort_values(by='coeficiente', ascending=False).head(40)
menores_coef = df_coef.sort_values(by='coeficiente').head(40)

plt.figure(figsize=(12, 8))
plt.barh(maiores_coef['feature'], maiores_coef['coeficiente'], color='green')
plt.title('40 Maiores Coeficientes (Positivo)')
plt.gca().invert_yaxis()
plt.show()

plt.figure(figsize=(12, 8))
plt.barh(menores_coef['feature'], menores_coef['coeficiente'], color='red')
plt.title('40 Menores Coeficientes (Negativo)')
plt.gca().invert_yaxis()
plt.show()